In [31]:

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [32]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier
import joblib


In [33]:
INPUT_PATH = "/content/drive/MyDrive/processed_dataset.xlsx"
df = pd.read_excel(INPUT_PATH)
print(df.shape)
df.head()



(10019, 36)


,order_id,supplier_id,supplier_rating,supplier_lead_time,order_date,promised_delivery_date,actual_delivery_date,shipping_distance_km,order_quantity,unit_price,...,region_South,region_West,holiday_period_Yes,carrier_name_DHL,carrier_name_Delhivery,carrier_name_EcomExpress,carrier_name_FedEx,delayed_reason_code_Operational,delayed_reason_code_Traffic,delayed_reason_code_Weather
0,1.0,5322.0,3.4,10,2024-05-15,2024-05-25,2024-05-29,51,48,2153.91,...,False,False,False,False,False,True,False,True,False,False
1,2.0,3932.0,4.3,10,2024-11-12,2024-11-22,2024-11-27,373,91,405.36,...,False,False,True,True,False,False,False,False,False,False
2,3.0,8966.0,3.2,5,2024-08-28,2024-09-02,2024-09-02,1304,25,3241.41,...,True,False,False,False,False,False,False,False,False,False
3,4.0,9832.0,3.9,7,2024-08-12,2024-08-19,2024-08-19,839,71,365.79,...,False,False,False,False,False,False,True,True,False,False
4,5.0,2126.0,3.2,8,2024-07-07,2024-07-15,2024-07-18,258,9,3052.84,...,False,False,False,False,False,False,False,False,False,False


In [34]:
df["on_time_delivery"] = (df["on_time_delivery"] == 1).astype(int)
print(df["on_time_delivery"].value_counts())


on_time_delivery
0    7239
1    2780
Name: count, dtype: int64


In [35]:
leakage_cols = [
    # IDs
    "order_id", "supplier_id",

    # Dates (post outcome)
    "order_date", "actual_delivery_date", "promised_delivery_date",

    # Outcome-derived
    "delivery_days",
    "delivery_speed",
    "delayed_reason_code_Operational",
    "delayed_reason_code_Traffic",
    "delayed_reason_code_Weather"
]

df.drop(columns=leakage_cols, errors="ignore", inplace=True)


In [36]:
df = df.loc[:, ~df.columns.duplicated()]
print("After cleanup:", df.shape)


After cleanup: (10019, 27)


In [37]:
df["long_distance"] = (df["shipping_distance_km"] > 1000).astype(int)
df["poor_supplier"] = (df["supplier_rating"] <= 3).astype(int)
df["low_history"] = (df["previous_on_time_rate"] < 60).astype(int)


In [38]:
df.drop(
    columns=[c for c in df.columns if c.startswith("delivery_speed_")],
    inplace=True,
    errors="ignore"
)


In [39]:
X = df.drop("on_time_delivery", axis=1)
y = df["on_time_delivery"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [40]:
scale_cols = [
    "supplier_lead_time",
    "shipping_distance_km",
    "order_quantity",
    "unit_price",
    "total_order_value",
    "previous_on_time_rate"
]

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[scale_cols] = scaler.fit_transform(X_train_scaled[scale_cols])
X_test_scaled[scale_cols] = scaler.transform(X_test_scaled[scale_cols])


In [41]:
lr = LogisticRegression(
    max_iter=5000,
    class_weight="balanced",
    solver="saga",
    n_jobs=-1
)

lr.fit(X_train_scaled, y_train)

lr_prob = lr.predict_proba(X_test_scaled)[:, 1]
lr_pred = (lr_prob >= 0.4).astype(int)


In [42]:
rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42
)

rf.fit(X_train, y_train)

rf_prob = rf.predict_proba(X_test)[:, 1]
rf_pred = (rf_prob >= 0.4).astype(int)


In [43]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    eval_metric="logloss",
    random_state=42
)

xgb.fit(X_train.values, y_train.values)

xgb_prob = xgb.predict_proba(X_test.values)[:, 1]
xgb_pred = (xgb_prob >= 0.4).astype(int)


In [44]:
def evaluate(name, y_true, y_pred, y_prob):
    print(f"\n📊 {name}")
    print(classification_report(y_true, y_pred))
    print("ROC-AUC:", roc_auc_score(y_true, y_prob))

evaluate("Logistic Regression", y_test, lr_pred, lr_prob)
evaluate("Random Forest", y_test, rf_pred, rf_prob)
evaluate("XGBoost", y_test, xgb_pred, xgb_prob)



📊 Logistic Regression
              precision    recall  f1-score   support

           0       1.00      0.00      0.00      1448
           1       0.28      1.00      0.43       556

    accuracy                           0.28      2004
   macro avg       0.64      0.50      0.22      2004
weighted avg       0.80      0.28      0.12      2004

ROC-AUC: 0.5110199133510871

📊 Random Forest
              precision    recall  f1-score   support

           0       0.72      0.98      0.83      1448
           1       0.37      0.03      0.06       556

    accuracy                           0.72      2004
   macro avg       0.55      0.51      0.44      2004
weighted avg       0.63      0.72      0.62      2004

ROC-AUC: 0.5106926199371996

📊 XGBoost
              precision    recall  f1-score   support

           0       0.74      0.30      0.43      1448
           1       0.28      0.72      0.41       556

    accuracy                           0.42      2004
   macro avg       0.

In [47]:
import joblib

# Save final trained model
joblib.dump(xgb, "best_model.pkl")

# Save scaler (used in deployment)
joblib.dump(scaler, "scaler.pkl")

# Save exact feature order
joblib.dump(X.columns.tolist(), "model_features.pkl")

print("✅ Model, scaler, and features saved successfully")


✅ Model, scaler, and features saved successfully
